# SNNAI v6-5-7 Single-GPU SNN LM + Transformer Comparison

This notebook runs a fast single-GPU training (5 epochs) to get results quickly.

- v6.5.0-v6.5.3: DataParallel attempts (incompatible with snntorch LIF)
- v6.5.4-v6.5.5: DDP attempts (mp.spawn hangs in Kaggle notebook)
- v6.5.7: Single-GPU inline training with more epochs with full corpus for better convergence

In [ ]:
# Install dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu126
!pip install -q snntorch numpy requests

In [ ]:
# Clone SNNAI repository
!rm -rf snnai
!git clone --depth 1 --branch v6.5.7 https://github.com/sabunosuke1008-create/snnai.git
import sys
sys.path.insert(0, 'snnai')

In [ ]:
# Detect GPU and set device
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    if cap[0] < 7:
        device = torch.device('cpu')
        print(f'GPU compute capability {cap} below 7.0; using CPU')
    else:
        device = torch.device('cuda')
        print('Using device:', device, torch.cuda.get_device_name(0))
else:
    device = torch.device('cpu')
    print('Using CPU')

In [ ]:
# Download corpus (Tiny Shakespeare + WikiText-2)
import requests
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
corpus = requests.get(url).text
try:
    from snnai.utils.download_corpus import download_wikitext2
    wt_root = download_wikitext2(dest_dir='/kaggle/working/wikitext-2', timeout=120)
    if wt_root is not None:
        wt_files = [
            str(wt_root) + '/wiki.train.raw',
            str(wt_root) + '/wiki.valid.raw',
            str(wt_root) + '/wiki.test.raw',
        ]
        wt_text = ''.join(open(f).read() for f in wt_files if __import__('os').path.exists(f))
        if wt_text:
            corpus = corpus + '\n' + wt_text
            print('Combined WikiText-2; corpus length:', len(corpus))
except Exception as exc:
    print('WikiText-2 download failed:', exc)
print('Final corpus length:', len(corpus))

In [ ]:
# Build BPE tokenizer (rank-0 style: just run once)
import torch.nn as nn
from snnai.modules.language import BPETokenizer, CharTokenizer
from snnai.modules.language.large_scale_lm import LargeScaleSNNLM
from snnai.benchmarks.transformer_comparison import TransformerBaseline

bpe = BPETokenizer([corpus], vocab_size=2048, max_train_bytes=2_000_000)
tokenizer = bpe
vocab_size = tokenizer.vocab_size
print('BPE vocab size:', vocab_size)

In [ ]:
# Model dimensions (moderate for fast single-GPU 5-epoch run)
embed_dim = 128
hidden_dim = 512
num_layers = 3
seq_len = 128
batch_size = 32
time_steps = 20
epochs = 5
lr = 5e-4
label_smoothing = 0.1

snn_model = LargeScaleSNNLM(vocab_size=vocab_size, embed_dim=embed_dim,
                            hidden_dim=hidden_dim, num_layers=num_layers,
                            dropout=0.2, output_mode='mem_last',
                            use_sequence_recurrent=True,
                            use_positional_encoding=True,
                            max_seq_len=seq_len).to(device)

transformer_model = TransformerBaseline(vocab_size=vocab_size, d_model=256,
                                        nhead=4, num_layers=4, dim_feedforward=1024).to(device)

In [ ]:
# Single-GPU inline training loop (3 epochs)
import math, time
from snnai.benchmarks.large_corpus_trainer import CharLMDataset, WarmupCosineSchedule
from snnai.benchmarks.homeostatic_loss import HomeostaticRegularizer

dataset = CharLMDataset(corpus, tokenizer, seq_len=seq_len)
val_size = int(len(dataset) * 0.05)
train_size = len(dataset) - val_size
from torch.utils.data import DataLoader, Subset
train_loader = DataLoader(Subset(dataset, list(range(train_size))),
                          batch_size=batch_size, shuffle=True, drop_last=True,
                          collate_fn=lambda b: __import__('snnai.benchmarks.large_corpus_trainer', fromlist=['collate_fn']).collate_fn(b, vocab_size))
val_loader = DataLoader(Subset(dataset, list(range(train_size, len(dataset)))),
                        batch_size=batch_size, shuffle=False, drop_last=False,
                        collate_fn=lambda b: __import__('snnai.benchmarks.large_corpus_trainer', fromlist=['collate_fn']).collate_fn(b, vocab_size))

snn_opt = torch.optim.AdamW(snn_model.parameters(), lr=lr, weight_decay=0.01)
tx_opt = torch.optim.AdamW(transformer_model.parameters(), lr=lr, weight_decay=0.01)
total_steps = epochs * len(train_loader)
snn_sched = WarmupCosineSchedule(snn_opt, warmup_steps=max(1, total_steps // 10),
                                 total_steps=max(1, total_steps), base_lr=lr, min_lr=1e-5)
tx_sched = WarmupCosineSchedule(tx_opt, warmup_steps=max(1, total_steps // 10),
                                total_steps=max(1, total_steps), base_lr=lr, min_lr=1e-5)
homeo = HomeostaticRegularizer(target_firing_rate=0.12, homeostatic_weight=1e-3)

history = {'snn': {'train_loss': [], 'val_loss': [], 'train_ppl': [], 'val_ppl': []},
           'transformer': {'train_loss': [], 'val_loss': [], 'train_ppl': [], 'val_ppl': []}}

for epoch in range(epochs):
    snn_model.train()
    snn_loss_sum, snn_tokens = 0.0, 0
    for one_hot, targets in train_loader:
        one_hot, targets = one_hot.to(device), targets.to(device)
        x = one_hot.unsqueeze(0).repeat(time_steps, 1, 1, 1)
        snn_opt.zero_grad()
        out, spikes = snn_model(x, return_spikes=True)
        ce = nn.functional.cross_entropy(out.reshape(-1, vocab_size), targets.reshape(-1),
                                       label_smoothing=label_smoothing, reduction='sum')
        homeo_loss = homeo(spikes)
        loss = ce + homeo_loss * targets.numel()
        loss.backward()
        nn.utils.clip_grad_norm_(snn_model.parameters(), 1.0)
        snn_opt.step()
        snn_sched.step()
        snn_loss_sum += ce.item()
        snn_tokens += targets.numel()
    snn_train_loss = snn_loss_sum / max(1, snn_tokens)

    transformer_model.train()
    tx_loss_sum, tx_tokens = 0.0, 0
    for one_hot, targets in train_loader:
        inputs = one_hot.argmax(dim=-1).to(device)
        targets = targets.to(device)
        tx_opt.zero_grad()
        out = transformer_model(inputs)
        loss = nn.functional.cross_entropy(out.reshape(-1, vocab_size), targets.reshape(-1),
                                       label_smoothing=label_smoothing, reduction='sum')
        loss.backward()
        nn.utils.clip_grad_norm_(transformer_model.parameters(), 1.0)
        tx_opt.step()
        tx_sched.step()
        tx_loss_sum += loss.item()
        tx_tokens += targets.numel()
    tx_train_loss = tx_loss_sum / max(1, tx_tokens)

    # Validation
    snn_model.eval()
    transformer_model.eval()
    with torch.no_grad():
        snn_v, snn_t = 0.0, 0
        tx_v, tx_t = 0.0, 0
        for one_hot, targets in val_loader:
            one_hot, targets = one_hot.to(device), targets.to(device)
            x = one_hot.unsqueeze(0).repeat(time_steps, 1, 1, 1)
            out, _ = snn_model(x, return_spikes=True)
            ce = nn.functional.cross_entropy(out.reshape(-1, vocab_size), targets.reshape(-1), reduction='sum')
            snn_v += ce.item()
            snn_t += targets.numel()
            inputs = one_hot.argmax(dim=-1)
            out = transformer_model(inputs)
            ce = nn.functional.cross_entropy(out.reshape(-1, vocab_size), targets.reshape(-1), reduction='sum')
            tx_v += ce.item()
            tx_t += targets.numel()
    snn_val_loss = snn_v / max(1, snn_t)
    tx_val_loss = tx_v / max(1, tx_t)
    snn_ppl = math.exp(min(20.0, snn_train_loss))
    snn_vppl = math.exp(min(20.0, snn_val_loss))
    tx_ppl = math.exp(min(20.0, tx_train_loss))
    tx_vppl = math.exp(min(20.0, tx_val_loss))
    print(f'Epoch {epoch}: SNN train_loss={snn_train_loss:.4f} ppl={snn_ppl:.2f} val_ppl={snn_vppl:.2f} | TX train_loss={tx_train_loss:.4f} ppl={tx_ppl:.2f} val_ppl={tx_vppl:.2f}')
    history['snn']['train_loss'].append(snn_train_loss)
    history['snn']['val_loss'].append(snn_val_loss)
    history['snn']['train_ppl'].append(snn_ppl)
    history['snn']['val_ppl'].append(snn_vppl)
    history['transformer']['train_loss'].append(tx_train_loss)
    history['transformer']['val_loss'].append(tx_val_loss)
    history['transformer']['train_ppl'].append(tx_ppl)
    history['transformer']['val_ppl'].append(tx_vppl)

# Save history
import json
with open('/kaggle/working/v6_5_7_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print('Saved /kaggle/working/v6_5_7_history.json')

# Save models
torch.save({'snn_state_dict': snn_model.state_dict(),
            'tx_state_dict': transformer_model.state_dict(),
            'vocab_size': vocab_size, 'history': history},
           '/kaggle/working/v6_5_7_models.pt')
print('Saved /kaggle/working/v6_5_7_models.pt')

In [ ]:
# Latency / parameter comparison
import time, json
from snnai.benchmarks.transformer_comparison import compare_models

sample_input = torch.zeros(time_steps, 1, seq_len, vocab_size).to(device)
comparison = compare_models(snn_model, transformer_model, sample_input)
print(json.dumps(comparison, indent=2))

# Energy report
from snnai.benchmarks.energy_quantification import quantize_energy
sample_text = 'ROMEO:'
sample_ids = tokenizer.encode(sample_text)[:seq_len]
sample_ids = sample_ids + [0] * (seq_len - len(sample_ids))
sample_one_hot = torch.zeros(1, seq_len, vocab_size)
sample_one_hot.scatter_(2, torch.tensor([sample_ids]).unsqueeze(2), 1.0)
sample_input_e = sample_one_hot.unsqueeze(0).repeat(time_steps, 1, 1, 1).to(device)
energy = quantize_energy(snn_model, sample_input_e, time_steps=time_steps)
print('\nSNN energy report:')
print(json.dumps(energy, indent=2))

In [ ]:
# Generate sample text
from snnai.benchmarks.generation_metrics import evaluate_generation
snn_model.eval()
sampled = evaluate_generation(snn_model, tokenizer, ['ROMEO:', 'JULIET:', 'The '],
                              max_chars=150, device=device,
                              temperature=0.8, top_k=10, top_p=0.9, do_sample=True,
                              repetition_penalty=1.5, penalty_window=16)
print('Sampled generation:')
for g in sampled['generated']:
    print(g[:150])